In [1]:
import csv
import os

In [2]:
# creazione della classe prodotto vegano

class VeganProduct:
    """
    Rappresenta un prodotto vegano con informazioni su quantità, prezzi e vendite.

    Attributi:
        product_name (str): nome del prodotto vegano.
        amount (int): quantità del prodotto vegano.
        purchase_price (float): prezzo di acquisto del prodotto vegano (in euro).
        selling_price (float): prezzo di vendita del prodotto vegano (in euro).
        sold (int): quantità del prodotto vegano venduta (default: 0).
    """

    
    def __init__(self, product_name, amount, purchase_price, selling_price, sold = 0):
        """
        Inizializza un nuovo oggetto VeganProduct con le informazioni specificate.

        Args:
            product_name (str): nome del prodotto vegano.
            amount (int): quantità del prodotto vegano.
            purchase_price (float): prezzo di acquisto del prodotto vegano (in euro).
            selling_price (float): prezzo di vendita del prodotto vegano (in euro).
            sold (int): quantità del prodotto vegano venduta (default: 0).
        """
        
        self.product_name = product_name
        self.amount = amount
        self.purchase_price = purchase_price
        self.selling_price = selling_price
        self.sold = sold



    def to_csv_row(self):
        """
        Restituisce i dati del prodotto come lista formattata per la scrittura su CSV.

        Args:
            Nessun argomento in input.

        Returns:
            list: Lista contenente [product_name, amount, purchase_price, selling_price, sold].
        """
       
        return [
            self.product_name,
            self.amount,
            self.purchase_price,
            self.selling_price,
            self.sold
        ]

    
    
    def __repr__(self):
        """
        Restituisce una rappresentazione leggibile dell'oggetto VeganProduct.

        Args:
            Nessun argomento in input.
        
        Returns:
            str: Descrizione testuale del prodotto con nome, quantità e prezzi.
        """
        
        return f"{self.product_name}, quantità {self.amount}, ha prezzo di acquisto €{self.purchase_price} e prezzo di vendita €{self.selling_price}."


In [3]:
# creazione della classe negozio di prodotti vegani

class VeganProductShop:
    """
    Rappresenta un negozio di prodotti vegani con gestione di magazzino e vendite.

    Attributi:
        filename (str): Nome del file CSV usato come magazzino.
        inventory (dict): Dizionario con chiave nome prodotto e valore oggetto VeganProduct.
    """


    
    def __init__(self, filename="magazzino_online.csv"):
        """
        Inizializza il magazzino del negozio con un magazzino vuoto
        caricando l’inventario dal file CSV o creando un nuovo file se assente.

        Args:
            filename (str, optional): Percorso del file CSV del magazzino. Default "magazzino_online.csv".
        """
        
        self.filename = filename
        self.inventory = {}
        self.load_inventory()


    
    def create_csv(self):
        """
        Crea il file CSV con intestazioni se non esiste già, cioè al primo avvio del programma.
        Una volta chiuso e riavviato il programma il CSV rimane salvato.

        Args:
            Nessun argomento in input.

        Side Effects:
            Crea il file CSV sul filesystem.
        """
        
        if not os.path.exists(self.filename):
            header = ["Nome del prodotto", "Quantità", "Prezzo di acquisto", "Prezzo di vendita", "Quantità vendute"]
            with open(self.filename, mode="w", newline="", encoding="utf-8") as file:
                writer = csv.writer(file)
                writer.writerow(header)


    
    def load_inventory(self):
        """
        Carica l’inventario dal file CSV nel dizionario `inventory`.
        
        Gestisce righe corrotte ignorandole e stampa messaggi di errore.

        Args:
            Nessun argomento in input.
    
        Side Effects:
            Aggiorna l’attributo `inventory` con i prodotti letti dal file CSV.
        """
        
        self.create_csv()
        try:
            with open(self.filename, newline="", encoding="utf-8") as file:
                reader = csv.DictReader(file)
                for row in reader:
                    try:
                        name = row["Nome del prodotto"]
                        amount = int(row["Quantità"])
                        purchase_price = float(row["Prezzo di acquisto"])
                        selling_price = float(row["Prezzo di vendita"])
                        sold = int(row["Quantità vendute"])
                        self.inventory[name] = VeganProduct(name, amount, purchase_price, selling_price, sold)
                    except (ValueError, KeyError) as e:
                        print(f"Riga del file CSV ignorata per errore: {e}")
                        continue
                        
        except Exception as e:
            print(f"Errore durante il caricamento dell'inventario: {e}")


    

    
    def save_inventory(self):
        """
        Salva l'inventario attuale sul file CSV.
        
        Il file CSV viene aperto in modalità "write" e viene interamente riscritto con tutti i prodotti presenti nell'inventario.
        IL file CSV non viene aperto in modalità "append" perchè così facendo, se si provasse ad aggiungere un prodotto già esistente
        (con la funzione add_product), verrebbe creata una nuova riga, invece di modificare la quantità del prodotto nella riga esistente.

        Args:
            Nessun argomento in input.

        Side Effects:
            Aggiorna o riscrive il file CSV `self.filename`.

        Raises:
            IOError: Se si verifica un errore di scrittura sul file.
        """
        
        try:
            header = ["Nome del prodotto", "Quantità", "Prezzo di acquisto", "Prezzo di vendita", "Quantità vendute"]
            # apre il file in modalità "write" e riscrive tutto il CSV
            with open(self.filename, mode="w", newline="", encoding="utf-8") as file:
                writer = csv.writer(file)
                writer.writerow(header)
                
                for prod in self.inventory.values():
                    writer.writerow(prod.to_csv_row())   
        
        except IOError:
            print("Errore nella scrittura del file CSV.\n")




    
    def add_product(self):
        """
        Aggiunge un prodotto al magazzino (CSV) nella quantità desiderata, con relativi prezzo di acquisto e di vendita
        o aggiorna la quantità di un prodotto già esistente nel magazzino.
        
        Funzionamento:
        - Chiede all’utente nome del prodotto e quantità.
        - Se il prodotto esiste, aggiorna la quantità.
        - Se il prodotto è nuovo, richiede i prezzi di acquisto e vendita.
        - Permette di annullare l’operazione digitando 'indietro'.
        - Salva automaticamente l’inventario aggiornato su file CSV.

        L’utente deve fornire l’input nel formato: "nome_prodotto, quantità".
        Se il prodotto è nuovo, sarà richiesto: "prezzo_acquisto, prezzo_vendita".

        Args:
            Nessun argomento in input. Richiede input utente durante l’esecuzione.

        Side Effects:
            Modifica il file CSV del magazzino.
            Stampa messaggi informativi a console.

        Raises:
            Nessuna eccezione viene propagata. Tutti gli errori sono gestiti internamente.
          
        Example:
            >>> add_product()
            Inserisci nome e quantità del prodotto separati da una virgola [nome del prodotto, quantità].
            Oppure 'indietro' per tornare al menu principale: 
             soia, 10
            Inserisci i prezzi di acquisto e di vendita del prodotto separati da una virgola [prezzo di acquisto, prezzo di vendita], 
            Oppure 'indietro' per tornare al menu principale: 
             1.2, 3.5
            Nome del prodotto: soia 
            Quantità: 10 
            Prezzo di acquisto: €1.2 
            Prezzo di vendita: €3.5 
            AGGIUNTO: 10 X soia 
        """
        
        while True: # continua a chiede all'utente di inserire l'input, finchè non ne viene inserito uno valido
            
            # Input dell'utente: nome e quantità del prodotto
            new_product = input((
                "Inserisci nome e quantità del prodotto separati da una virgola [nome del prodotto, quantità].\n"
                "Oppure 'indietro' per tornare al menu principale: \n"
            ))
            
            if new_product.strip().lower() == "indietro":
                print("Operazione annullata. Ritorno al menu principale.\n")
                return
            
            parts = new_product.split(",")
    
            # Controlla che il formato dell'input sia corretto (due elementi separati da virgola)
            if len(parts) != 2:
                print("Formato errato. \n \n")
                continue
        
            name = parts[0].strip()
    
            # Controlla che la quantità sia un numero intero
            try:
                added_amount = int(parts[1].strip())
                if added_amount <= 0:
                    raise ValueError
            except ValueError:
                print("Quantità non valida. La quantità deve essere un numero intero maggiore di 0. \n \n")
                continue
                
            break # Input valido → esce dal ciclo
 
    
        if name in self.inventory:
            # Prodotto già esistente: aggiorna quantità
            self.inventory[name].amount += added_amount
            
            # considera l'oggetto VeganProduct associato a 'name' e accede ai suoi attributi
            product = self.inventory[name] 
            purchase_price = product.purchase_price
            selling_price = product.selling_price
            
            print(f"Nome del prodotto: {name} \n"
                  f"Quantità: {added_amount} \n"
                  f"Prezzo di acquisto: €{purchase_price} \n"
                  f"Prezzo di vendita: €{selling_price} \n"
                  f"AGGIUNTO: {added_amount} X {name} \n\n")
            
        else:
            # Nuovo prodotto: chiedi prezzi
            while True: # continua a chiede all'utente di inserire l'input, finchè non ne viene inserito uno valido
                prices = input((
                    "Inserisci i prezzi di acquisto e di vendita del prodotto appena aggiunto separati da una virgola [prezzo di acquisto, prezzo di vendita], \n"
                    "Oppure 'indietro' per tornare al menu principale: \n"
                ))
            
                if prices.strip().lower() == "indietro":
                    print("Operazione annullata. Ritorno al menu principale.\n")
                    return
                    
                price_parts = prices.split(",")
    
                if len(price_parts) != 2:
                    print("Formato errato. \n")
                    continue
    
                try:
                    purchase_price = float(price_parts[0].strip())
                    selling_price = float(price_parts[1].strip())

                    # Crea il nuovo prodotto
                    self.inventory[name] = VeganProduct(name, added_amount, purchase_price, selling_price)
                    
                    print(f"Nome del prodotto: {name} \n"
                          f"Quantità: {added_amount} \n"
                          f"Prezzo di acquisto: €{purchase_price} \n"
                          f"Prezzo di vendita: €{selling_price} \n"
                          f"AGGIUNTO: {added_amount} X {name} \n\n")
                    
                    break  # Input validoi → esce dal ciclo
                    
                except ValueError:
                    print("Prezzi non validi. Usa numeri decimali con punto (esempio: 3.5). \n")


        # Salva l'inventario aggiornato
        self.save_inventory()





    def list_products(self):
        """
        Elenca tutti i prodotti presenti nel magazzino con relativi quantità e prezzi di vendita.
        Stampa una tabella formattata direttamente a console.
        
        Args:
            Nessun argomento in input.
            
        Side Effects:
            Stampa direttamente a console una tabella formattata con i prodotti.
    
        Raises:
            Nessuna eccezione viene sollevata direttamente da questo metodo.
    
        Example:
            >>> list_products()
            PRODOTTO            QUANTITÀ    PREZZO
            Latte di soia       10          €1.39
            Riso basmati        20          €2.40    
        """
        
        if not self.inventory:
            print("Il magazzino è vuoto.")
            return

        print("\nPRODOTTO".ljust(20) + "QUANTITÀ".ljust(10) + "PREZZO VENDITA")
        for product in self.inventory.values():
            print(f"{product.product_name.ljust(20)}{str(product.amount).ljust(10)}€{product.selling_price:.2f}")
        print("")





    def record_sale(self):
        """
        Registra una o più vendite aggiornando quantità e vendite nel magazzino.
        
        Funzionamento:
        - L’utente inserisce nome del prodotto e quantità da vendere.
        - per ogni vendita, verifica disponibilità del prodotto e aggiorna l'inventario.
        - Le vendite vengono salvate temporaneamente.
        - L’utente può confermare la vendita o annullare l’intera operazione digitando 'indietro'.
        - Solo dopo la conferma l’inventario reale e il CSV vengono aggiornati.
        - Stampa il riepilogo e il totale delle vendite confermate.
    
        Args:
            Nessun argomento in input. Richiede input utente durante l’esecuzione.
            
        Side Effects:
            Aggiorna e salva il file CSV del magazzino.
            Stampa riepilogo vendite e totale a console.
    
        Raises:
            Nessuna eccezione viene propagata. Gli errori di input sono gestiti durante l’esecuzione.

        Example:
            >>> record_sale()
            Inserisci nome e quantità del prodotto separati da virgola [nome del prodotto, quantità], 
            Oppure 'indietro' per tornare al menu principale: 
             tofu, 2
            Aggiungere un altro prodotto ? ('si, 'no', oppure 'indietro' per tornare al menu principale):  si
            Inserisci nome e quantità del prodotto separati da virgola [nome del prodotto, quantità], 
            Oppure 'indietro' per tornare al menu principale: 
             miele, 1
            Aggiungere un altro prodotto ? ('si, 'no', oppure 'indietro' per tornare al menu principale):  no
            
            VENDITA REGISTRATA
            - 2 X tofu: €6.80
            - 1 X miele: €5.00
            Totale: €11.80
        """
        
        if not self.inventory: # gestisce il caso in cui il magazzino è vuoto
            print("Il magazzino è vuoto. Nessun prodotto da vendere.")
            return

        temp_inventory = {name: product.amount for name, product in self.inventory.items()}
        sales_list = []  # lista per tenere traccia delle vendite
        total = 0.0  # totale da calcolare alla fine
    
        # utilizzo un while loop che continua a chiedere in input prodotti da vendere finchè l'utente non vuole fermarsi.
        # in ogni momento all'utente è data la possibilità di tornare al menu principale.
        while True:
            # Input dell'utente: nome e quantità del prodotto
            sold_product = input((
                "Inserisci nome e quantità del prodotto separati da virgola [nome del prodotto, quantità], \n"
                "Oppure 'indietro' per annullare l'intera operazione: \n"))
            
            if sold_product.strip().lower() == "indietro":
                print("Operazione annullata. Ritorno al menu principale.\n")
                return
            
            sold_product_parts = sold_product.split(",")
    
            # Controlla che il formato dell'input sia corretto (due elementi separati da virgola)
            if len(sold_product_parts) != 2:
                print("Formato errato. \n \n")
                continue
            #rimuovo spazi all'inizio e alla fine del nome del prodotto
            sold_product_name = sold_product_parts[0].strip()
    
            # Controlla che la quantità sia un numero intero
            try:
                sold_amount = int(sold_product_parts[1].strip())
                if sold_amount <= 0:
                    raise ValueError
            except ValueError:
                print("Quantità non valida. La quantità deve essere un numero intero maggiore di 0. \n \n")
                continue

            
            if sold_product_name not in self.inventory:
                print("Prodotto non trovato.\n")
                continue


            if temp_inventory[sold_product_name] < sold_amount:
                print(f"Quantità insufficiente in magazzino: {temp_inventory[sold_product_name]} disponibili.\n")
                continue

            
            # Aggiorna solo la copia temporanea
            temp_inventory[sold_product_name] -= sold_amount
            sales_list.append((sold_product_name, sold_amount))
            total += sold_amount * self.inventory[sold_product_name].selling_price
        
           
            # Continua a chiede altri prodotti da vendere
            while True:
                another_product = input("Aggiungere un altro prodotto ? ('si, 'no', oppure 'indietro' per annullare l'intera operazione): ").strip().lower()
    
                if another_product == "si":
                    break  # esce dal ciclo interno e ripete il ciclo esterno
    
                elif another_product == "no":
                    # Conferma la vendita: aggiorna inventario reale
                    for name, amount in sales_list:
                        self.inventory[name].amount -= amount
                        self.inventory[name].sold += amount
                    self.save_inventory()
                    print("\nVENDITA REGISTRATA")
                   
                    for name, amount in sales_list:
                        partial = amount * self.inventory[name].selling_price
                        print(f"- {amount} X {name}: €{partial:.2f}")
                    print(f"Totale: €{total:.2f}\n")
                    return
 
                elif another_product == "indietro":
                    print("Operazione annullata. Nessuna vendita registrata.\n")
                    return
    
                else:
                    print("Comando non valido, scegli 'si' o 'no' o 'indietro'. \n")
        





    def show_profits(self):
        """
        Calcola e mostra i profitti lordi e netti.

        Definizioni:
        - Profitto lordo: somma degli incassi totali dalle vendite.
          Il profitto lordo è calcolato come: quantità venduta * prezzo di vendita per ciascun prodotto.
        - Profitto netto: profitto lordo meno il costo di acquisto dei prodotti venduti.
          Il profitto netto è calcolato come: profitto lordo meno il costo di acquisto per i prodotti

        Args:
            Nessun argomento in input.
        
        Side Effects:
            Stampa a console il profitto lordo totale.
            Stampa a console il profitto netto totale.
            
        Raises
            Nessuna eccezione viene sollevata da questo metodo.
    
        Example:
            >>> show_profits()
            Profitto lordo totale: €251.20
            Profitto netto totale: €164.60
        """
    
    
        # Profitto lordo
        gross_profit = 0
        tot_purchase = 0    

        for product in self.inventory.values():
            gross_profit += product.sold * product.selling_price
            tot_purchase += product.sold * product.purchase_price
            
        print(f"\nProfitto lordo totale: €{gross_profit:.2f}")
        
        # Profitto netto
        net_profit = gross_profit - tot_purchase
        print(f"Profitto netto totale: €{net_profit:.2f}\n")
      
    

    def show_commands(self):
        """
        Mostra a video l’elenco dei comandi disponibili per l'utente.
    
        Args:
            Nessun argomento in input.
        
        Side Effects:
            Stampa a console l’elenco dei comandi disponibili.
            
        Raises:
            Non può sollevare alcuna eccezione, poichè non prende input, non elabora dati, 
            e non interagisce con file o elementi esterni.
    
        Example:
            >>> show_commands()
            I comandi disponibili sono i seguenti:
            aggiungi: aggiungi un prodotto al magazzino
            elenca: elenca i prodotto in magazzino
            vendita: registra una vendita effettuata
            profitti: mostra i profitti totali
            aiuto: mostra i possibili comandi
            chiudi: esci dal programma        
        """
        
        print("I comandi disponibili sono i seguenti:")
        print("aggiungi: aggiungi un prodotto al magazzino")
        print("elenca: elenca i prodotti in magazzino")
        print("vendita: registra una vendita effettuata")
        print("profitti: mostra i profitti totali")
        print("aiuto: mostra i possibili comandi")
        print("chiudi: esci dal programma \n")





    def menu(self):
        """
        Mostra il menu principale del programma e gestisce i comando dell'utente.
    
        Il menu è interattivo e funziona in modalità ciclo continuo fino a quando
        l’utente non inserisce il comando "chiudi".
    
        Comandi disponibili:
            - aggiungi: aggiunge un prodotto o aggiorna uno esistente nel magazzino
            - elenca: stampa l'elenco dei prodotti disponibili
            - vendita: registra una o più vendite
            - profitti: mostra il totale dei profitti generati
            - aiuto: mostra l’elenco dei comandi
            - chiudi: termina il programma
        
        Args:
            Nessun argomento in input. Richiesta input utente durante l’esecuzione.
        
        Side Effects:
            Interagisce con l’utente tramite input/output.
            Aggiorna il file CSV a seconda dei comandi.
    
        Raises:
            Nessuna eccezione viene propagata. Tutte le eccezioni sono gestite tramite blocco try-except.
        
        Example:
            >>> menu()
            I comandi disponibili sono i seguenti:
            aggiungi: aggiungi un prodotto al magazzino
            elenca: elenca i prodotti in magazzino
            vendita: registra una vendita effettuata
            profitti: mostra i profitti totali
            aiuto: mostra i possibili comandi
            chiudi: esci dal programma 
            Inserisci un comando: ___
        """
        # richiama la funzione che mostra i comandi, perchè l'utente al primo approccio 
        # con il programma potrebbe non sapere quali comandi può inserire
        self.show_commands() 
        
        while True:
            command = input("Inserisci un comando:")
    
            try:
                if command == "aggiungi":
                    self.add_product()
                elif command == "elenca":
                    self.list_products()
                elif command == "vendita":
                    self.record_sale()
                elif command == "profitti":
                    self.show_profits()
                elif command == "aiuto":
                    self.show_commands()
                elif command == "chiudi":
                    print("Programma terminato.")
                    break
                else:
                    print("Comando non valido, riprova.\n\n")
            
            except Exception as e:
                print(f"Si è verificato un errore imprevisto: {e}\n")


In [4]:
# Avvio del programma
if __name__ == "__main__":
    shop = VeganProductShop()
    shop.menu()

I comandi disponibili sono i seguenti:
aggiungi: aggiungi un prodotto al magazzino
elenca: elenca i prodotti in magazzino
vendita: registra una vendita effettuata
profitti: mostra i profitti totali
aiuto: mostra i possibili comandi
chiudi: esci dal programma 



Inserisci un comando: elenca



PRODOTTO           QUANTITÀ  PREZZO VENDITA
seitan              5         €1.40
tofu                48        €2.00
pichi               0         €1000000.00
mele                101       €2.00



Inserisci un comando: vendita
Inserisci nome e quantità del prodotto separati da virgola [nome del prodotto, quantità], 
Oppure 'indietro' per annullare l'intera operazione: 
 mele, 50
Aggiungere un altro prodotto ? ('si, 'no', oppure 'indietro' per annullare l'intera operazione):  tofu, 5


Comando non valido, scegli 'si' o 'no' o 'indietro'. 



...

I comandi disponibili sono i seguenti:
aggiungi: aggiungi un prodotto al magazzino
elenca: elenca i prodotti in magazzino
vendita: registra una vendita effettuata
profitti: mostra i profitti totali
aiuto: mostra i possibili comandi
chiudi: esci dal programma 



Inserisci un comando: profitti



Profitto lordo totale: €1000530.00
Profitto netto totale: €470.50



Inserisci un comando: chiudi


Programma terminato.
